# Random Forest Crime Prediction Assignment

## Overview
In this assignment, you will use a **recent real-world crime dataset** to build a **Random Forest classification model**. You will explore the data, prepare it for modeling, train a classifier, evaluate the results, and interpret feature importance.

## Learning Goals
By the end of this assignment, you should be able to:

- explain the difference between **features** and a **target variable**
- prepare data for machine learning
- build a **Random Forest** classifier in Python
- evaluate a classification model using accuracy, a confusion matrix, and a classification report
- interpret **feature importance**
- connect model results to real-world patterns and possible limitations

## Important Note
In this notebook, you are **not predicting whether a crime will happen**.  
You are predicting:

> **Was an arrest made?** (`True` or `False`)

That means your model is learning patterns related to **arrest outcomes**, not crime occurrence itself.


## Step 1: Load the Data

We will use a recent subset of the Chicago crime dataset. The dataset is large, so we will pull a manageable sample for class use.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$limit=50000"
df = pd.read_csv(url)

df.head()

## Step 2: Explore the Data

Look at the structure of the dataset.

### Questions
1. What does each row represent? Each row represents one reported crime incident in Chicago, including details such as the type of crime, whether an arrest was made, location, date, and other attributes.
2. Which variables look potentially useful for prediction? Useful variables include: primary_type (crime type), domestic, district, latitude, longitude, and especially arrest (the target). Other columns like beat, ward, or community_area could also be useful but we are limiting the features for this assignment.
3. Which variable appears to be the target for this assignment? The target variable is arrest (True/False), because the assignment asks us to predict whether an arrest was made for the reported incident.


In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## Step 3: Data Cleaning

We will keep a smaller set of variables for this assignment.

- `primary_type`: type of crime
- `arrest`: whether an arrest was made
- `domestic`: whether the incident was domestic-related
- `district`: police district
- `latitude`, `longitude`: location

Then we will remove rows with missing values.


In [ ]:
df = df[['primary_type', 'arrest', 'domestic', 'district', 'latitude', 'longitude']]
df = df.dropna()

df.head()

### Short Response
Why might missing values cause problems in machine learning and mapping?
Missing values can cause problems because:

Many machine learning algorithms (including Random Forest) cannot handle NaN values directly and will either fail to train or produce incorrect results.
For mapping, missing latitude or longitude makes it impossible to plot the point on a map, leading to errors or blank visualizations.
Dropping rows with missing values (as we did) ensures the data is clean and consistent for both modeling and visualization.

## Step 4: Interactive Crime Map

Now that the data has been cleaned, we can build a map using latitude and longitude.

We are placing this **after data cleaning** because missing values in location fields can cause mapping tools to fail.

### Note
We will use `CartoDB positron` for the base map. This tile option often works better in notebook environments than the default map tiles.


In [ ]:
import folium

m = folium.Map(
    location=[df['latitude'].mean(), df['longitude'].mean()],
    zoom_start=11,
    tiles="CartoDB positron"
)

sample_df = df.sample(1000, random_state=42)

for _, row in sample_df.iterrows():
    color = "red" if row['arrest'] else "blue"
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2,
        color=color,
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

m

### Interpretation Questions
1. Do crime points appear evenly spread out, or do they cluster in certain areas? Crime points are not evenly spread. They tend to cluster heavily in certain neighborhoods and districts (especially on the South and West Sides of Chicago), while other areas (e.g., downtown or northern neighborhoods) show much lower density.
2. Do you see any visible differences between arrest and non-arrest locations? There is some visible difference. Red dots (arrests) and blue dots (no arrest) are mixed, but arrests appear somewhat more common in certain high-crime clusters. However, the pattern is not extremely obvious from the map alone — this is why we need the model to quantify it.
3. If location turns out to be important later, does this map help explain why? Yes. The map shows strong geographic clustering of crimes. If location features (latitude, longitude, or district) end up being highly important, it is likely because arrest practices, police presence, or crime types vary significantly by neighborhood.


## Step 5: Arrest Rate by Crime Type

This chart helps you see which crime categories are more likely to lead to an arrest.


In [ ]:
arrest_rates = df.groupby('primary_type')['arrest'].mean().sort_values(ascending=False)

plt.figure(figsize=(10,6))
arrest_rates.head(10).plot(kind='bar')
plt.title("Top 10 Crime Types by Arrest Rate")
plt.ylabel("Arrest Rate")
plt.xlabel("Crime Type")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Interpretation Question
Which crime types seem most likely to lead to arrest? Does this surprise you?
The crime types with the highest arrest rates are typically:

Narcotics (often > 90%)
Gambling
Prostitution
Weapons violations / Liquor law violations
Some violent crimes like Homicide or Battery (varies)

This is not very surprising. “In-progress” or easily verifiable crimes like drug possession, weapons offenses, or prostitution usually result in immediate arrests because evidence is present at the scene. In contrast, property crimes like theft or burglary often have much lower arrest rates because they are reported after the fact and witnesses/suspects are harder to identify.

## Step 6: Feature Engineering

Machine learning models need numeric input. Since `primary_type` and `district` are categorical, we will convert them into dummy variables.


In [ ]:
df = pd.get_dummies(df, columns=['primary_type', 'district'], drop_first=True)
df.head()

## Step 7: Define Features and Target

Here is the key setup:

- `X` = the input features used to make predictions
- `y` = the target variable we want to predict

In this assignment:

> `y = arrest`

So the model predicts whether an arrest was made.


In [ ]:
X = df.drop('arrest', axis=1)
y = df['arrest']

## Step 8: Train/Test Split

We split the data so the model trains on one portion and is evaluated on unseen data.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Step 9: Train the Random Forest Model


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

## Step 10: Evaluate the Model

We will use:
- **Accuracy**
- **Confusion Matrix**
- **Classification Report**


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Step 11: Feature Importance

Feature importance tells us which variables had the biggest influence on the model’s predictions.

A larger value means the model used that variable more heavily when making decisions.


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(10)
top_features

## Step 12: Visualize Feature Importance

This plot gives a clearer picture of what the model relied on most.


In [ ]:
plt.figure(figsize=(8,6))
top_features.sort_values().plot(kind='barh')
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

    ### Interpretation Questions
1. Which features matter the most? Which features matter the most?
Usually the top features are:
latitude and longitude (location)
Certain primary_type categories (especially Narcotics, Weapons Violation, etc.)
domestic (sometimes)
Specific district dummies
2. Does the model seem to rely more on **location** or **crime type**? The model typically relies more on location (latitude + longitude) and specific crime types than on domestic or most individual districts. Location often ranks very high because arrest likelihood varies significantly by neighborhood (due to differences in policing, crime density, etc.).
3. What does that suggest about arrest outcomes in the dataset? It suggests that where a crime occurs and what type of crime it is are much stronger predictors of whether an arrest is made than other factors. This reflects real-world patterns: some neighborhoods and crime categories have systematically higher (or lower) clearance/arrest rates.


## Step 13: Cross Validation

Cross validation helps us check whether the model performs consistently across different subsets of the data.


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X, y, cv=5)
print("Cross-validation scores:", scores)
print("Average cross-validation score:", scores.mean())

## Step 14: Reflection

Write a short reflection that answers the following:

1. What does the model do well? The Random Forest model does a good job of predicting whether an arrest was made, achieving decent accuracy and capturing important patterns in the data (especially through location and crime type).
2. What are its limitations? The model only predicts arrest outcomes, not whether a crime actually occurred or who committed it.
It can inherit biases present in the historical data (e.g., over-policing in certain neighborhoods).
With only a sample of 50,000 records and limited features, it may not generalize perfectly to all situations or future data.
Random Forest can be prone to overfitting if not properly tuned.
3. What does feature importance suggest about the strongest predictors of arrest? Feature importance shows that geographic location (latitude/longitude) and the type of crime (primary_type) are by far the strongest predictors. This indicates that arrest decisions are heavily influenced by where the incident happened and the nature of the offense (e.g., narcotics offenses are much more likely to result in arrest than theft).
4. Why is it important to remember that this model predicts **arrest outcomes** rather than crime itself? It is crucial because arrest ≠ crime occurrence. Many crimes go unsolved (no arrest), and arrest rates can reflect policing priorities, resources, or biases rather than the actual distribution of crime. Using the model to “predict crime” would be misleading and potentially harmful.
5. Should a model like this be used in real-world policing decisions? Why or why not? No, it should not be used directly for operational policing decisions. While it can help identify patterns for research or resource allocation, relying on it risks reinforcing existing biases, over-policing certain neighborhoods, and violating fairness principles. Any real-world use would require careful ethical review, bias auditing, transparency, and human oversight. Predictive models in criminal justice must prioritize justice and equity, not just accuracy.
